# LLM Quality Monitoring with Amazon CloudWatch and Amazon Managed Grafana

This notebook demonstrates end-to-end LLM quality monitoring for Amazon SageMaker AI inference components:

## Workflow Overview
1. **Invoke Amazon SageMaker AI inference components** - Send requests to deployed LLMs
2. **Log to CloudWatch Logs** - Capture requests, responses, and metadata
3. **Calculate Quality Scores** - Use LLM-as-judge (Claude via Amazon Bedrock) to evaluate responses
4. **Publish Custom Metrics** - Send scores to CloudWatch Metrics
5. **Create Grafana Dashboards** - Visualize quality metrics over time
6. **Provision Grafana Alerts** - Deploy threshold-based alert rules for quality regressions

## Quality Metrics Tracked
- **Safety**: Content safety and policy compliance
- **Relevance**: How well answers address the query
- **Coherence**: Logical flow and consistency
- **Professional Tone**: Appropriate communication style
- **No Harmful Advice**: Absence of dangerous recommendations
- **Latency**: Response time tracking

## Prerequisites
- An active Amazon SageMaker AI endpoint with one or more inference components
- An Amazon SageMaker AI Managed MLflow App or tracking server (**MLflow 3.4 or later** required for tracing and judge orchestration)
- Amazon Bedrock model access for the LLM-as-judge evaluator (default: Anthropic Claude Sonnet 4) in your region, or cross-region inference enabled
- AWS IAM Identity Center (SSO) configured in the account — used by Amazon Managed Grafana for user authentication
- The caller's AWS credentials must have at minimum the following IAM permissions:
  - `cloudwatch:PutMetricData`, `cloudwatch:ListMetrics`
  - `logs:CreateLogGroup`, `logs:PutLogEvents`, `logs:PutRetentionPolicy`
  - `bedrock:InvokeModel`
  - `grafana:CreateWorkspace`, `grafana:DescribeWorkspace`, `grafana:UpdateWorkspaceConfiguration`, `grafana:CreateWorkspaceApiKey`
  - `iam:CreateRole`, `iam:PutRolePolicy`, `iam:UpdateAssumeRolePolicy`
  - `sns:CreateTopic`, `sns:Subscribe` (only if `ALERT_NOTIFICATION_EMAIL` is configured)

## Cost

This notebook creates billable AWS resources, including an Amazon Managed Grafana workspace, Amazon CloudWatch log groups and custom metrics, and (if `ALERT_NOTIFICATION_EMAIL` is set) an Amazon SNS topic. You will incur charges while these resources exist. Run **Step 14: Clean Up Resources** at the end of the notebook to delete them and stop the charges when you are done.

## Configuration

Edit the single cell below to set your deployment-specific values. Everything else
in the notebook references these variables — you should not need to edit any other cell.

| Variable | Description |
|---|---|
| `ENDPOINT_NAME` | Your Amazon SageMaker AI endpoint name |
| `INFERENCE_COMPONENT_1` | First inference component name |
| `INFERENCE_COMPONENT_1_LABEL` | Friendly label for the first IC |
| `INFERENCE_COMPONENT_2` | Second inference component name |
| `INFERENCE_COMPONENT_2_LABEL` | Friendly label for the second IC |
| `MLFLOW_TRACKING_APP_URI` | MLflow 3.4+ tracking server / app ARN |
| `ALERT_NOTIFICATION_EMAIL` | Email to subscribe to the alert SNS topic (leave the placeholder to skip SNS setup) |

`REGION` and `ACCOUNT_ID` are auto-detected from your environment.

**Note on MLflow version:** The tracking server/app must be on **MLflow 3.4+**.

## Step 1: Setup and Configuration


In [ ]:
# Install required dependencies
!pip install boto3 requests mlflow==3.8.1 --quiet --upgrade


In [ ]:
# ===================== EDIT THESE VALUES =====================
  
ENDPOINT_NAME               = "<your-sagemaker-endpoint-name>"
INFERENCE_COMPONENT_1       = "<your-inference-component-1-name>"
INFERENCE_COMPONENT_1_LABEL = "<model-1-label>"
INFERENCE_COMPONENT_2       = "<your-inference-component-2-name>"
INFERENCE_COMPONENT_2_LABEL = "<model-2-label>"
MLFLOW_TRACKING_APP_URI     = "arn:aws:sagemaker:<region>:<account-id>:mlflow-app/<app-id>"

# Email address to receive Grafana alert notifications via SNS.
# Leave as-is to skip SNS setup; you can configure a contact point later in the Grafana UI.
ALERT_NOTIFICATION_EMAIL    = "<your-email@example.com>"

# ==============================================================

In [ ]:
import boto3
import json
import os
import time
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any
import mlflow

# ── Auto-detect region & account ──
REGION = boto3.Session().region_name
ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# ── Build the inference-components list from the flat variables above ──
INFERENCE_COMPONENTS = [
    {"name": INFERENCE_COMPONENT_1, "label": INFERENCE_COMPONENT_1_LABEL},
    {"name": INFERENCE_COMPONENT_2, "label": INFERENCE_COMPONENT_2_LABEL},
]


_creds = boto3.Session().get_credentials()
if _creds is None:
    raise RuntimeError(
        "No AWS credentials found. Configure them via 'aws configure', "
        "AWS_PROFILE, or run from an environment with an attached IAM role."
    )
_frozen = _creds.get_frozen_credentials()
os.environ["AWS_ACCESS_KEY_ID"]     = _frozen.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = _frozen.secret_key
if _frozen.token:
    os.environ["AWS_SESSION_TOKEN"] = _frozen.token
os.environ["AWS_REGION"]            = REGION
os.environ.pop("AWS_ROLE_ARN", None)
del _creds, _frozen  # don't leak the temporary helpers into the global namespace

# ── AWS clients (created once, reused everywhere) ──
sm_runtime       = boto3.client("sagemaker-runtime", region_name=REGION)
cloudwatch_logs  = boto3.client("logs",              region_name=REGION)
cloudwatch_metrics = boto3.client("cloudwatch",      region_name=REGION)
bedrock_runtime  = boto3.client("bedrock-runtime",   region_name=REGION)
iam              = boto3.client("iam")
grafana          = boto3.client("grafana",           region_name=REGION)

# Back-compat aliases used by some cells
region = REGION
account_id = ACCOUNT_ID

# ── Fixed / derived configuration ──
LOG_GROUP_PREFIX           = "/aws/sagemaker/inference-quality"
METRIC_NAMESPACE           = "SageMaker/InferenceQuality"
MLFLOW_EVALUATION_MODEL_ID = "bedrock:/global.anthropic.claude-sonnet-4-6"
MLFLOW_EVALUATION_MODEL_PARAM = {
    "temperature": 0, "max_tokens": 512,
    "anthropic_version": "bedrock-2023-05-31",
    "top_p": 0.9, "stop_sequences": ["}"],
}
THRESHOLDS = {
    "safety_min": 0.8, "relevance_min": 0.7,
    "coherence_min": 0.75, "latency_max_ms": 5000, "composite_min": 0.75,
}
GRAFANA_WORKSPACE_NAME  = "llm-quality-monitoring"
GRAFANA_ROLE_NAME       = "AmazonGrafanaCloudWatchRole"
GRAFANA_DATASOURCE_NAME = "CloudWatch-Quality"
GRAFANA_DASHBOARD_UID   = "llm-quality-monitoring"
GRAFANA_DASHBOARD_TITLE = "LLM Quality Monitoring - SageMaker"

# ── MLflow tracking ──
mlflow.set_tracking_uri(MLFLOW_TRACKING_APP_URI)
mlflow.set_experiment("default")

# ── Summary ──
print(f"✅ Region: {REGION}  Account: {ACCOUNT_ID}  MLflow: {mlflow.__version__}")
print(f"   Endpoint: {ENDPOINT_NAME}")
for ic in INFERENCE_COMPONENTS:
    print(f"   IC: {ic['label']} → {ic['name']}")
print(f"   Tracking: {MLFLOW_TRACKING_APP_URI}")

## Step 2: CloudWatch Logs Setup

Create log groups for each inference component to store request/response data.

In [ ]:
def create_log_group(ic_name: str) -> str:
    """Create CloudWatch log group for inference component."""
    log_group_name = f"{LOG_GROUP_PREFIX}/{ic_name}"
    
    try:
        cloudwatch_logs.create_log_group(logGroupName=log_group_name)
        print(f"✅ Created log group: {log_group_name}")
    except cloudwatch_logs.exceptions.ResourceAlreadyExistsException:
        print(f"ℹ️  Log group exists: {log_group_name}")
    
    # Set retention to 7 days
    try:
        cloudwatch_logs.put_retention_policy(
            logGroupName=log_group_name,
            retentionInDays=7
        )
    except Exception as e:
        print(f"⚠️  Could not set retention: {e}")
    
    return log_group_name

# Create log groups
log_groups = {}
for ic in INFERENCE_COMPONENTS:
    log_groups[ic['name']] = create_log_group(ic['name'])

print(f"\n✅ CloudWatch Logs setup complete!")

## Step 3: Inference and Logging Functions

Functions to invoke endpoints and log to CloudWatch.

In [ ]:
def invoke_inference_component(prompt: str, ic_name: str, endpoint_name: str) -> Dict[str, Any]:
    """Invoke Amazon SageMaker AI inference component and measure latency."""
    payload = {
        'inputs': prompt,
        'parameters': {
            'max_new_tokens': 512,
            'temperature': 0.7,
            'top_p': 0.9,
            'do_sample': True
        }
    }
    
    start_time = time.time()
    
    try:
        response = sm_runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            InferenceComponentName=ic_name,
            ContentType='application/json',
            Body=json.dumps(payload)
        )
        
        latency_ms = (time.time() - start_time) * 1000
        response_body = json.loads(response['Body'].read().decode())
        
        # Extract generated text (format may vary by model)
        if isinstance(response_body, list) and len(response_body) > 0:
            generated_text = response_body[0].get('generated_text', '')
        elif 'generated_text' in response_body:
            generated_text = response_body['generated_text']
        elif 'choices' in response_body:
            generated_text = response_body['choices'][0]['message']['content']
        else:
            generated_text = str(response_body)
        
        return {
            'success': True,
            'prompt': prompt,
            'response': generated_text,
            'latency_ms': latency_ms,
            'timestamp': datetime.utcnow().isoformat(),
            'ic_name': ic_name
        }
    
    except Exception as e:
        return {
            'success': False,
            'prompt': prompt,
            'error': str(e),
            'latency_ms': (time.time() - start_time) * 1000,
            'timestamp': datetime.utcnow().isoformat(),
            'ic_name': ic_name
        }

In [ ]:
def log_to_cloudwatch(log_group_name: str, log_data: Dict[str, Any]):
    """Log inference data to CloudWatch Logs."""
    log_stream_name = datetime.utcnow().strftime('%Y/%m/%d/quality-monitoring')
    
    # Create log stream if it doesn't exist
    try:
        cloudwatch_logs.create_log_stream(
            logGroupName=log_group_name,
            logStreamName=log_stream_name
        )
    except cloudwatch_logs.exceptions.ResourceAlreadyExistsException:
        pass
    
    # Put log event
    try:
        cloudwatch_logs.put_log_events(
            logGroupName=log_group_name,
            logStreamName=log_stream_name,
            logEvents=[
                {
                    'timestamp': int(time.time() * 1000),
                    'message': json.dumps(log_data)
                }
            ]
        )
    except Exception as e:
        print(f"⚠️  Failed to log: {e}")

## Step 4: MLflow Quality Evaluation Scorers

Configure MLflow scorers for LLM-as-judge quality evaluation using Amazon Bedrock.

In [ ]:
import mlflow
from mlflow.genai.scorers import Safety, RelevanceToQuery, Guidelines#, Fluency #, make_judge

# Configure MLflow scorers for quality evaluation
safety_scorer = Safety(
    model=MLFLOW_EVALUATION_MODEL_ID, 
    parameters=MLFLOW_EVALUATION_MODEL_PARAM
)

relevance_scorer = RelevanceToQuery(
    model=MLFLOW_EVALUATION_MODEL_ID, 
    parameters=MLFLOW_EVALUATION_MODEL_PARAM
)

professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="The response must be in a professional tone.",
    model=MLFLOW_EVALUATION_MODEL_ID,
    parameters=MLFLOW_EVALUATION_MODEL_PARAM
)

# Combine all scorers
quality_scorers = [
    safety_scorer,
    relevance_scorer,
    professional_tone_scorer,
]

print("✅ MLflow quality scorers configured:")
print(f"   - Safety (built-in)")
print(f"   - RelevanceToQuery (built-in)")
print(f"   - Professional Tone (Guidelines-based)")

## Step 5: Publish Custom Metrics to CloudWatch

Send quality scores as custom CloudWatch metrics.

In [ ]:
def publish_quality_metrics(ic_name: str, ic_label: str, scores: Dict[str, float], latency_ms: float):
    """Publish quality scores as CloudWatch custom metrics."""
    metric_data = []
    timestamp = datetime.utcnow()
    
    # Common dimensions
    dimensions = [
        {'Name': 'InferenceComponentName', 'Value': ic_name},
        {'Name': 'InferenceComponentLabel', 'Value': ic_label},
        {'Name': 'EndpointName', 'Value': ENDPOINT_NAME}
    ]
    
    # Add quality score metrics
    for metric_name, value in scores.items():
        metric_data.append({
            'MetricName': metric_name,
            'Dimensions': dimensions,
            'Value': value,
            'Timestamp': timestamp,
            'Unit': 'None',
            'StorageResolution': 60  # High-resolution metric (1-minute)
        })
    
    # Add latency metric
    metric_data.append({
        'MetricName': 'quality_evaluation_latency',
        'Dimensions': dimensions,
        'Value': latency_ms,
        'Timestamp': timestamp,
        'Unit': 'Milliseconds',
        'StorageResolution': 60
    })
    
    # Composite quality score (average of all scores)
    composite_score = sum(scores.values()) / len(scores)
    metric_data.append({
        'MetricName': 'composite_quality_score',
        'Dimensions': dimensions,
        'Value': composite_score,
        'Timestamp': timestamp,
        'Unit': 'None',
        'StorageResolution': 60
    })
    
    # Publish metrics in batches (max 1000 per call)
    try:
        cloudwatch_metrics.put_metric_data(
            Namespace=METRIC_NAMESPACE,
            MetricData=metric_data
        )
    except Exception as e:
        print(f"⚠️  Failed to publish metrics: {e}")

print("✅ Metric publishing function ready!")

## Step 6: End-to-End Quality Monitoring Pipeline

Complete pipeline: invoke → log → evaluate → publish metrics.

In [ ]:
def monitor_inference_quality(prompt: str, ic_config: Dict[str, str]) -> Dict[str, Any]:
    """Complete quality monitoring pipeline for a single inference using MLflow evaluation."""
    ic_name = ic_config['name']
    ic_label = ic_config['label']
    
    print(f"\n🔄 Processing: {ic_label}")
    
    # Step 1: Invoke inference component
    print("  1️⃣ Invoking inference component...")
    inference_result = invoke_inference_component(prompt, ic_name, ENDPOINT_NAME)
    
    if not inference_result['success']:
        print(f"  ❌ Inference failed: {inference_result.get('error')}")
        return inference_result
    
    print(f"  ✅ Response received ({inference_result['latency_ms']:.0f}ms)")
    response_text = inference_result['response']
    
    # Step 2: Log request/response to CloudWatch Logs
    print("  2️⃣ Logging to CloudWatch Logs...")
    log_to_cloudwatch(log_groups[ic_name], {
        'type': 'inference',
        'timestamp': inference_result['timestamp'],
        'prompt': prompt,
        'response': response_text,
        'latency_ms': inference_result['latency_ms']
    })
    
    # Step 3: Evaluate quality using MLflow
    print("  3️⃣ Evaluating quality with MLflow scorers...")
    
    try:
        eval_dataset = [
            {
                "inputs": {"question": prompt},
                "outputs": response_text,
            }
        ]
        
        eval_results = mlflow.genai.evaluate(
            data=eval_dataset,
            scorers=quality_scorers,
        )
        
        # Extract scores from evaluation results
        # MLflow returns aggregate metrics like 'safety/mean', 'relevance_to_query/mean', etc.
        mlflow_metrics = eval_results.metrics
        
        # Map MLflow scorer results to CloudWatch metric names (normalize the names)
        quality_scores = {
            'safety_score': float(mlflow_metrics.get('safety/mean', 0)),
            'relevance_score': float(mlflow_metrics.get('relevance_to_query/mean', 0)),
            'professional_tone_score': float(mlflow_metrics.get('professional_tone/mean', 0))
        }
        
        print(f"  📊 Quality Scores:")
        for metric, score in quality_scores.items():
            print(f"     {metric}: {score:.3f}")
    
    except Exception as e:
        error_msg = f"MLflow evaluation failed: {e}"
        print(f"  ❌ {error_msg}")
        # Raise error instead of using fallback scores
        raise RuntimeError(error_msg)
    
    # Step 4: Log quality scores to CloudWatch Logs
    print("  4️⃣ Logging quality scores...")
    log_to_cloudwatch(log_groups[ic_name], {
        'type': 'quality_evaluation',
        'timestamp': datetime.utcnow().isoformat(),
        'scores': mlflow_metrics  # Log original MLflow metrics
    })
    
    # Step 5: Publish metrics to CloudWatch Metrics
    print("  5️⃣ Publishing metrics to CloudWatch...")
    publish_quality_metrics(ic_name, ic_label, quality_scores, inference_result['latency_ms'])
    
    print(f"  ✅ Complete for {ic_label}")
    
    return {
        **inference_result,
        'quality_scores': quality_scores
    }

print("✅ Quality monitoring pipeline ready (with MLflow GenAI evaluation format)!")

## Step 7: Test the Pipeline

Run quality monitoring on test prompts.

In [ ]:
test_prompts = [
    "Explain machine learning in simple terms.",
    "What is the difference between supervised and unsupervised learning?",
    "How do neural networks work?",
    "What are transformers in deep learning?",
    "Explain the concept of gradient descent."
]

print(f"🚀 Running quality monitoring on {len(test_prompts)} prompts...\n")
print(f"   Testing {len(INFERENCE_COMPONENTS)} inference components")
print(f"   Total inferences: {len(test_prompts) * len(INFERENCE_COMPONENTS)}")
print("="*80)

results = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*80}")
    print(f"Prompt {i}/{len(test_prompts)}: {prompt[:60]}...")
    print(f"{'='*80}")
    
    for ic_config in INFERENCE_COMPONENTS:
        result = monitor_inference_quality(prompt, ic_config)
        results.append(result)
        
        # Small delay to avoid rate limiting
        time.sleep(2)

print(f"\n\n{'='*80}")
print(f"✅ QUALITY MONITORING COMPLETE")
print(f"{'='*80}")
print(f"   Total inferences: {len(results)}")
print(f"   Successful: {sum(1 for r in results if r['success'])}")
print(f"   Failed: {sum(1 for r in results if not r['success'])}")
print(f"\n📊 Metrics published to CloudWatch namespace: {METRIC_NAMESPACE}")
print(f"📝 Logs available in CloudWatch log groups under: {LOG_GROUP_PREFIX}")


## Step 8: Create Grafana Workspace (if not exists)

Set up Amazon Managed Grafana for visualization and alerting.

The workspace is created via service-managed permissions with AWS IAM Identity Center (SSO) auth and CloudWatch as a data source. After the workspace is `ACTIVE`, the cell also enables **unified alerting** if it isn't already enabled — this is required for the Step 11 alert rules to actually evaluate. If unified alerting is currently off, the cell will turn it on and wait for the workspace to restart (~5 min).

In [ ]:
# Grafana IAM role (service-managed). Uses the iam client defined at the top.

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "grafana.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"aws:SourceAccount": ACCOUNT_ID}},
    }],
}

try:
    iam.create_role(
        RoleName=GRAFANA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
    )
    print(f"✅ Created IAM role: {GRAFANA_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"ℹ️  IAM role exists: {GRAFANA_ROLE_NAME}")

# SECURITY NOTE — accepted risk: Resource: "*" on CloudWatch read APIs
#
# The CloudWatch metric APIs (DescribeAlarmsForMetric, DescribeAlarms,
# ListMetrics, GetMetricData, GetMetricStatistics) and several Logs read APIs
# do NOT support resource-level permissions per the IAM Reference for Amazon
# CloudWatch and Amazon CloudWatch Logs. Granting these actions therefore
# requires Resource: "*". This is a documented AWS service constraint, not
# a policy oversight. For tighter control, layer additional scoping outside
# the policy itself: scope the Amazon Managed Grafana workspace to a small
# group of operators via IAM Identity Center, and use service control
# policies (SCPs) at the AWS Organizations level to bound CloudWatch access
# to specific accounts.
# Reference: https://docs.aws.amazon.com/service-authorization/latest/reference/list_amazoncloudwatch.html
cloudwatch_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": [
            "cloudwatch:DescribeAlarmsForMetric",
            "cloudwatch:DescribeAlarms",
            "cloudwatch:ListMetrics",
            "cloudwatch:GetMetricData",
            "cloudwatch:GetMetricStatistics",
            "logs:DescribeLogGroups",
            "logs:GetLogGroupFields",
            "logs:StartQuery",
            "logs:StopQuery",
            "logs:GetQueryResults",
        ],
        "Resource": "*",
    }],
}

iam.put_role_policy(
    RoleName=GRAFANA_ROLE_NAME,
    PolicyName="CloudWatchReadAccess",
    PolicyDocument=json.dumps(cloudwatch_policy),
)

grafana_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GRAFANA_ROLE_NAME}"
print(f"✅ IAM role configured: {grafana_role_arn}")

In [ ]:
# Check for an existing Grafana workspace and reuse if found; otherwise create one.
existing = [
    w for w in grafana.list_workspaces()["workspaces"]
    if "sagemaker-monitoring" in w["name"] or GRAFANA_WORKSPACE_NAME in w["name"]
]

if existing:
    workspace_id = existing[0]["id"]
    print(f"ℹ️  Using existing Grafana workspace: {existing[0]['name']}")
    print(f"   Workspace ID: {workspace_id}")
else:
    print("🔄 Creating new Grafana workspace...")
    resp = grafana.create_workspace(
        workspaceName=GRAFANA_WORKSPACE_NAME,
        accountAccessType="CURRENT_ACCOUNT",
        authenticationProviders=["AWS_SSO"],
        permissionType="SERVICE_MANAGED",
        workspaceRoleArn=grafana_role_arn,
        workspaceDataSources=["CLOUDWATCH"],
    )
    workspace_id = resp["workspace"]["id"]
    print(f"   Workspace ID: {workspace_id}")
    print("   Waiting for workspace to become active (5-10 min)...")
    while True:
        status = grafana.describe_workspace(workspaceId=workspace_id)["workspace"]["status"]
        print(f"   Status: {status}")
        if status == "ACTIVE":
            break
        time.sleep(15)

workspace = grafana.describe_workspace(workspaceId=workspace_id)["workspace"]
grafana_url = f"https://{workspace['endpoint']}"

print(f"\n✅ Grafana workspace ready!")
print(f"   URL: {grafana_url}")
print(f"   Workspace ID: {workspace_id}")

# ── Enable unified alerting on the workspace if not already enabled ──
# Amazon Managed Grafana workspaces can ship with unified alerting disabled. When it's off,
# alert rules can be defined via the provisioning API but Grafana never
# evaluates them (lastEvaluation stays at the zero timestamp). The Step 11
# alert rules below depend on this being on.
#
# Toggling this setting restarts the workspace (~5 min) and is destructive
# to existing alerting config (rules, contact points, policies). The block
# below skips the toggle when it's already enabled, so re-runs are no-ops.
ws_cfg_resp   = grafana.describe_workspace_configuration(workspaceId=workspace_id)
ws_cfg        = json.loads(ws_cfg_resp["configuration"])
alerting_on   = ws_cfg.get("unifiedAlerting", {}).get("enabled", False)

if alerting_on:
    print("✅ Unified alerting already enabled.")
else:
    print("⚠️  Unified alerting is OFF — enabling it now.")
    print("   This restarts the workspace (~5 min) and will RESET any")
    print("   existing alert rules / contact points / notification policies.")
    new_cfg = {**ws_cfg, "unifiedAlerting": {"enabled": True}}
    grafana.update_workspace_configuration(
        workspaceId=workspace_id,
        configuration=json.dumps(new_cfg),
        grafanaVersion=ws_cfg_resp.get("grafanaVersion", workspace["grafanaVersion"]),
    )
    print("   Waiting for workspace to return to ACTIVE...")
    while True:
        s = grafana.describe_workspace(workspaceId=workspace_id)["workspace"]["status"]
        print(f"   Status: {s}")
        if s == "ACTIVE":
            break
        time.sleep(15)
    print("✅ Unified alerting enabled.")

## Step 9: Configure Grafana Data Source and Dashboard

In [ ]:
# Create a short-lived admin API key for the workspace.
api_key_resp = grafana.create_workspace_api_key(
    workspaceId=workspace_id,
    keyName=f"quality-dashboard-{int(time.time())}",
    keyRole="ADMIN",
    secondsToLive=7200,
)
api_key = api_key_resp["key"]

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

# Reuse an existing CloudWatch data source if we already created one.
print("🔍 Checking for existing CloudWatch data sources...")
existing_datasources = requests.get(
    f"{grafana_url}/api/datasources",
    headers=headers,
).json()

cloudwatch_datasource = next(
    (
        ds for ds in existing_datasources
        if ds.get("name") == GRAFANA_DATASOURCE_NAME and ds.get("type") == "cloudwatch"
    ),
    None,
)

if cloudwatch_datasource:
    ds_uid = cloudwatch_datasource["uid"]
    print(f"ℹ️  CloudWatch data source already exists: {cloudwatch_datasource['name']}")
    print(f"   Using existing UID: {ds_uid}")
else:
    print("🔄 Creating new CloudWatch data source...")
    datasource_resp = requests.post(
        f"{grafana_url}/api/datasources",
        headers=headers,
        json={
            "name": GRAFANA_DATASOURCE_NAME,
            "type": "cloudwatch",
            "access": "proxy",
            "isDefault": True,
            "jsonData": {
                "authType": "default",
                "defaultRegion": REGION,
            },
        },
    ).json()

    if "datasource" in datasource_resp:
        ds_uid = datasource_resp["datasource"]["uid"]
    elif "uid" in datasource_resp:
        ds_uid = datasource_resp["uid"]
    else:
        raise ValueError(f"Failed to create data source: {datasource_resp}")
    print(f"✅ CloudWatch data source created: {ds_uid}")

print(f"\n✅ CloudWatch data source configured with UID: {ds_uid}")


## Step 10: Create LLM Quality Dashboard Panels

In [ ]:
def create_quality_metric_panel(panel_id: int, title: str, metric_name: str, 
                                 y_pos: int, x_pos: int = 0, width: int = 12) -> dict:
    """Create a Grafana panel for quality metrics."""
    return {
        "id": panel_id,
        "type": "timeseries",
        "title": title,
        "gridPos": {"h": 8, "w": width, "x": x_pos, "y": y_pos},
        "datasource": {"type": "cloudwatch", "uid": "${DS_CLOUDWATCH}"},
        "targets": [
            {
                "refId": f"A_{ic['label']}",
                "namespace": METRIC_NAMESPACE,
                "metricName": metric_name,
                "dimensions": {
                    "InferenceComponentName": [ic['name']],
                    "InferenceComponentLabel": [ic['label']],
                    "EndpointName": [ENDPOINT_NAME]
                },
                "statistic": "Average",
                "period": "60",
                "label": ic['label'],
                "region": region,
                "matchExact": True
            }
            for ic in INFERENCE_COMPONENTS
        ],
        "fieldConfig": {
            "defaults": {
                "custom": {
                    "drawStyle": "line",
                    "lineInterpolation": "smooth",
                    "fillOpacity": 10,
                    "lineWidth": 2,
                    "thresholdsStyle": {"mode": "dashed"}
                },
                "unit": "percentunit",
                "min": 0,
                "max": 1,
                "thresholds": {
                    "mode": "absolute",
                    "steps": [
                        {"color": "red", "value": None},
                        {"color": "yellow", "value": 0.7},
                        {"color": "green", "value": 0.85}
                    ]
                }
            }
        },
        "options": {
            "legend": {"displayMode": "list", "placement": "bottom"},
            "tooltip": {"mode": "multi"}
        }
    }

# Create dashboard panels
panels = []
panel_id = 1

# Row 1: Composite Score and Safety
panels.append(create_quality_metric_panel(
    panel_id, "Composite Quality Score", "composite_quality_score", 0, 0, 12
))
panel_id += 1

panels.append(create_quality_metric_panel(
    panel_id, "Safety Score", "safety_score", 0, 12, 12
))
panel_id += 1

# Row 2: Relevance and Professional Tone
panels.append(create_quality_metric_panel(
    panel_id, "Relevance Score", "relevance_score", 8, 0, 12
))
panel_id += 1

panels.append(create_quality_metric_panel(
    panel_id, "Professional Tone Score", "professional_tone_score", 8, 12, 12
))
panel_id += 1

# Row 3: Latency
latency_panel = {
    "id": panel_id,
    "type": "timeseries",
    "title": "Quality Evaluation Latency",
    "gridPos": {"h": 8, "w": 24, "x": 0, "y": 16},
    "datasource": {"type": "cloudwatch", "uid": "${DS_CLOUDWATCH}"},
    "targets": [
        {
            "refId": f"A_{ic['label']}",
            "namespace": METRIC_NAMESPACE,
            "metricName": "quality_evaluation_latency",
            "dimensions": {
                "InferenceComponentName": [ic['name']],
                "InferenceComponentLabel": [ic['label']],
                "EndpointName": [ENDPOINT_NAME]
            },
            "statistic": "Average",
            "period": "60",
            "label": ic['label'],
            "region": region,
            "matchExact": True
        }
        for ic in INFERENCE_COMPONENTS
    ],
    "fieldConfig": {
        "defaults": {
            "custom": {
                "drawStyle": "line",
                "lineInterpolation": "smooth",
                "fillOpacity": 10,
                "lineWidth": 2
            },
            "unit": "ms"
        }
    },
    "options": {
        "legend": {"displayMode": "list", "placement": "bottom"},
        "tooltip": {"mode": "multi"}
    }
}
panels.append(latency_panel)

print(f"✅ Created {len(panels)} dashboard panels")
print(f"   Panels created for metrics:")
print(f"     - composite_quality_score")
print(f"     - safety_score")
print(f"     - relevance_score")
print(f"     - professional_tone_score")
print(f"     - quality_evaluation_latency")

In [ ]:
# Deploy the dashboard. UID / title come from the shared config cell.
dashboard_payload = {
    "dashboard": {
        "uid": GRAFANA_DASHBOARD_UID,
        "title": GRAFANA_DASHBOARD_TITLE,
        "tags": ["sagemaker", "llm", "quality", "monitoring"],
        "timezone": "browser",
        "refresh": "1m",
        "templating": {
            "list": [{
                "name": "DS_CLOUDWATCH",
                "type": "datasource",
                "query": "cloudwatch",
                "current": {"text": GRAFANA_DATASOURCE_NAME, "value": ds_uid},
                "hide": 0,
                "label": "CloudWatch Data Source",
            }],
        },
        "panels": panels,
    },
    "overwrite": True,
}

dashboard_resp = requests.post(
    f"{grafana_url}/api/dashboards/db",
    headers=headers,
    json=dashboard_payload,
)

dashboard_result = dashboard_resp.json()
dashboard_url = f"{grafana_url}{dashboard_result['url']}"

print(f"\n✅ Dashboard deployed successfully!")
print(f"   Dashboard URL: {dashboard_url}")
print(f"   Status: {dashboard_result['status']}")


## Step 11: Provision Grafana Alerts

Deploy threshold-based alert rules via the Grafana Alerting Provisioning API. Each rule fires when its metric's most-recent average falls below the configured threshold for at least 5 minutes.

**Note:** Alert notifications require a contact point (Email, Slack, SNS, etc.). After the cell runs, configure one in the Grafana UI at `/alerting/notifications`.

In [ ]:
# Provision alert rules via the Grafana Alerting Provisioning API.
# Creates a folder + a rule group with three threshold-based rules.
# Re-runnable: existing rules are updated rather than duplicated.

ALERT_FOLDER_UID   = "llm-quality-alerts"
ALERT_FOLDER_TITLE = "LLM Quality Alerts"
ALERT_RULE_GROUP   = "llm-quality-rules"

# X-Disable-Provenance keeps rules editable in the Grafana UI;
# without it, API-provisioned rules are read-only there.
provisioning_headers = {**headers, "X-Disable-Provenance": "true"}

# ── 1. Create / reuse the alert folder ──
folder_resp = requests.post(
    f"{grafana_url}/api/folders",
    headers=headers,
    json={"uid": ALERT_FOLDER_UID, "title": ALERT_FOLDER_TITLE},
)
if folder_resp.status_code in (200, 201):
    print(f"✅ Created alert folder: {ALERT_FOLDER_UID}")
elif folder_resp.status_code in (409, 412):
    print(f"ℹ️  Alert folder exists: {ALERT_FOLDER_UID}")
else:
    raise RuntimeError(
        f"Folder creation failed: {folder_resp.status_code} {folder_resp.text}"
    )


# ── 2. Helper: build a Grafana alert-rule payload ──
# Grafana 10+ requires a 3-step expression chain:
#   A: CloudWatch query → time series
#   B: reduce(A) → scalar (one value per series)
#   C: threshold(B < N) → boolean (the alert condition)
# Without the explicit reduce step, Grafana flags the rule as 'error:
# invalid format of evaluation results — only reduced data can be alerted on'.
def build_alert_rule(uid: str, title: str, metric_name: str, threshold: float,
                     severity: str, summary: str) -> dict:
    """Threshold-below-N alert that queries CloudWatch every minute."""
    return {
        "uid": uid,
        "title": title,
        "ruleGroup": ALERT_RULE_GROUP,
        "folderUID": ALERT_FOLDER_UID,
        "condition": "C",  # C is the threshold expression that gates B
        "data": [
            # A: query CloudWatch for the metric (returns a time series)
            {
                "refId": "A",
                "queryType": "",
                "relativeTimeRange": {"from": 600, "to": 0},
                "datasourceUid": ds_uid,
                "model": {
                    "refId": "A",
                    "datasource": {"type": "cloudwatch", "uid": ds_uid},
                    "namespace": METRIC_NAMESPACE,
                    "metricName": metric_name,
                    # Wildcard on InferenceComponentName so Grafana receives one
                    # series per IC with a unique label. Without a labelled
                    # dimension Grafana's reduce step fails with 'frame cannot
                    # uniquely be identified by its labels'.
                    "dimensions": {"InferenceComponentName": "*"},
                    "statistic": "Average",
                    "period": "60",
                    "region": REGION,
                    "matchExact": False,
                    "metricQueryType": 0,
                    "metricEditorMode": 0,
                },
            },
            # B: reduce the time series to a single scalar (most-recent value)
            {
                "refId": "B",
                "queryType": "",
                "relativeTimeRange": {"from": 0, "to": 0},
                "datasourceUid": "__expr__",
                "model": {
                    "refId": "B",
                    "type": "reduce",
                    "datasource": {"type": "__expr__", "uid": "__expr__"},
                    "expression": "A",
                    "reducer": "last",
                    "settings": {"mode": "dropNN"},  # drop NaN/None values
                },
            },
            # C: threshold check — fires when the scalar is below the limit
            {
                "refId": "C",
                "queryType": "",
                "relativeTimeRange": {"from": 0, "to": 0},
                "datasourceUid": "__expr__",
                "model": {
                    "refId": "C",
                    "type": "threshold",
                    "datasource": {"type": "__expr__", "uid": "__expr__"},
                    "expression": "B",
                    "conditions": [{
                        "evaluator": {"params": [threshold], "type": "lt"},
                        "operator": {"type": "and"},
                        "query":    {"params": ["B"]},
                        "reducer":  {"type": "last", "params": []},
                        "type": "query",
                    }],
                },
            },
        ],
        "noDataState": "NoData",
        "execErrState": "Alerting",
        "for": "5m",
        "annotations": {
            "description": f"{summary} (threshold < {threshold})",
            "summary": summary,
        },
        "labels": {"severity": severity, "metric": metric_name},
        "isPaused": False,
        "orgID": 1,
    }


alert_rules = [
    build_alert_rule(
        "safety-score-alert", "Low Safety Score Alert", "safety_score",
        THRESHOLDS["safety_min"], "critical", "Low safety score detected",
    ),
    build_alert_rule(
        "relevance-score-alert", "Low Relevance Score Alert", "relevance_score",
        THRESHOLDS["relevance_min"], "warning", "Low relevance score detected",
    ),
    build_alert_rule(
        "composite-quality-alert", "Low Composite Quality Score Alert",
        "composite_quality_score", THRESHOLDS["composite_min"], "warning",
        "Low overall quality detected",
    ),
]


# ── 3. Create or update each alert rule (idempotent) ──
for rule in alert_rules:
    existing = requests.get(
        f"{grafana_url}/api/v1/provisioning/alert-rules/{rule['uid']}",
        headers=provisioning_headers,
    )
    if existing.status_code == 200:
        resp = requests.put(
            f"{grafana_url}/api/v1/provisioning/alert-rules/{rule['uid']}",
            headers=provisioning_headers,
            json=rule,
        )
        action = "Updated"
    else:
        resp = requests.post(
            f"{grafana_url}/api/v1/provisioning/alert-rules",
            headers=provisioning_headers,
            json=rule,
        )
        action = "Created"

    if resp.status_code in (200, 201):
        print(f"✅ {action} alert: {rule['title']}")
    else:
        raise RuntimeError(
            f"Failed to {action.lower()} {rule['title']}: "
            f"{resp.status_code} {resp.text}"
        )

print(f"\n✅ Alert rules deployed!")
print(f"   Folder:     {ALERT_FOLDER_TITLE}")
print(f"   Rule group: {ALERT_RULE_GROUP}")
print(f"   View at:    {grafana_url}/alerting/list")
print(f"\nℹ️  To receive notifications, configure a contact point in Grafana UI:")
print(f"   {grafana_url}/alerting/notifications")
print(f"   Supported in Managed Grafana: SNS, Slack, PagerDuty, OpsGenie, VictorOps.")
print(f"   For email: create an SNS topic with email subscribers and use the SNS contact point.")

## Step 12: Configure SNS Notifications (optional)

Provision the notification path for the alert rules deployed in Step 11. Amazon Managed Grafana doesn't ship an SMTP server, so email notifications are routed through Amazon SNS:

1. Create / reuse an SNS topic
2. Subscribe `ALERT_NOTIFICATION_EMAIL` to the topic
3. Grant the Grafana workspace IAM role permission to publish to the topic
4. Create a Grafana SNS contact point that auths via the workspace role
5. Add a notification-policy route so alerts in the `LLM Quality Alerts` folder go to that contact point

**You will receive an SNS subscription confirmation email from AWS — choose the link to activate it.** Until confirmed, alerts publish to SNS but no email is delivered.

If `ALERT_NOTIFICATION_EMAIL` is left at the placeholder, this cell is a no-op and you can configure a contact point manually at `/alerting/notifications`.

In [ ]:
# Provision an SNS topic + Grafana contact point so quality alerts deliver
# to a real inbox.

if not ALERT_NOTIFICATION_EMAIL or ALERT_NOTIFICATION_EMAIL.startswith("<"):
    print("ℹ️  SNS notifications not configured (ALERT_NOTIFICATION_EMAIL unset).")
    print(f"   Set ALERT_NOTIFICATION_EMAIL in the config cell to enable, then re-run.")
    print(f"   Or configure a contact point manually at: {grafana_url}/alerting/notifications")
else:
    sns = boto3.client("sns", region_name=REGION)
    SNS_TOPIC_NAME      = "llm-quality-alerts"
    CONTACT_POINT_UID   = "llm-quality-sns"
    CONTACT_POINT_NAME  = "LLM Quality Notifications"

    # ── 1. Create / reuse the SNS topic (boto3 create_topic is idempotent) ──
    sns_topic_arn = sns.create_topic(Name=SNS_TOPIC_NAME)["TopicArn"]
    print(f"✅ SNS topic: {sns_topic_arn}")

    # ── 2. Subscribe the email address (skip if already subscribed) ──
    existing_subs = sns.list_subscriptions_by_topic(
        TopicArn=sns_topic_arn,
    )["Subscriptions"]
    has_email = any(
        s["Protocol"] == "email" and s["Endpoint"] == ALERT_NOTIFICATION_EMAIL
        for s in existing_subs
    )
    if has_email:
        print(f"ℹ️  Email already subscribed: {ALERT_NOTIFICATION_EMAIL}")
    else:
        sns.subscribe(
            TopicArn=sns_topic_arn,
            Protocol="email",
            Endpoint=ALERT_NOTIFICATION_EMAIL,
        )
        print(f"✅ Subscribed: {ALERT_NOTIFICATION_EMAIL}")
        print(f"   ⚠️  Confirm the subscription via the email AWS just sent")
        print(f"      (Subject: 'AWS Notification - Subscription Confirmation').")

    # ── 3. Grant the Grafana workspace role permission to publish ──
    iam.put_role_policy(
        RoleName=GRAFANA_ROLE_NAME,
        PolicyName="GrafanaSNSPublish",
        PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Action": "sns:Publish",
                "Resource": sns_topic_arn,
            }],
        }),
    )
    print(f"✅ Granted sns:Publish on the topic to {GRAFANA_ROLE_NAME}")

    # ── 4. Create / update the Grafana SNS contact point ──
    contact_point = {
        "uid": CONTACT_POINT_UID,
        "name": CONTACT_POINT_NAME,
        "type": "sns",
        "settings": {
            "topic": sns_topic_arn,
            "authProvider": "default",  # use the workspace IAM role
            "messageFormat": "json",
        },
        "disableResolveMessage": False,
    }

    existing_cps = requests.get(
        f"{grafana_url}/api/v1/provisioning/contact-points",
        headers=provisioning_headers,
    ).json()

    if any(cp.get("uid") == CONTACT_POINT_UID for cp in existing_cps):
        cp_resp = requests.put(
            f"{grafana_url}/api/v1/provisioning/contact-points/{CONTACT_POINT_UID}",
            headers=provisioning_headers,
            json=contact_point,
        )
        cp_action = "Updated"
    else:
        cp_resp = requests.post(
            f"{grafana_url}/api/v1/provisioning/contact-points",
            headers=provisioning_headers,
            json=contact_point,
        )
        cp_action = "Created"

    if cp_resp.status_code in (200, 201, 202):
        print(f"✅ {cp_action} Grafana contact point: {CONTACT_POINT_NAME}")
    else:
        raise RuntimeError(
            f"Contact point setup failed: {cp_resp.status_code} {cp_resp.text}"
        )

    # ── 5. Add a notification-policy route for our alert folder ──
    policy = requests.get(
        f"{grafana_url}/api/v1/provisioning/policies",
        headers=provisioning_headers,
    ).json()

    routes = [
        r for r in policy.get("routes", [])
        if r.get("receiver") != CONTACT_POINT_NAME
    ]
    routes.append({
        "receiver": CONTACT_POINT_NAME,
        "object_matchers": [["grafana_folder", "=", ALERT_FOLDER_TITLE]],
        "group_by": ["alertname", "metric"],
    })
    policy["routes"] = routes

    pol_resp = requests.put(
        f"{grafana_url}/api/v1/provisioning/policies",
        headers=provisioning_headers,
        json=policy,
    )
    if pol_resp.status_code in (200, 201, 202):
        print(f"✅ Notification policy routes '{ALERT_FOLDER_TITLE}' alerts "
              f"to {CONTACT_POINT_NAME}")
    else:
        raise RuntimeError(
            f"Policy update failed: {pol_resp.status_code} {pol_resp.text}"
        )

    print(f"\n✅ Email notifications configured!")
    print(f"   Topic ARN:  {sns_topic_arn}")
    print(f"   Email:      {ALERT_NOTIFICATION_EMAIL}")
    print(f"   Confirm:    Choose the link in the AWS Notifications subscription email")

## Step 13: View Results and Summary

In [ ]:
# Summary statistics
successful_results = [r for r in results if r["success"]]

if successful_results:
    print("\n" + "=" * 80)
    print("📊 QUALITY MONITORING SUMMARY")
    print("=" * 80)

    for ic in INFERENCE_COMPONENTS:
        ic_results = [r for r in successful_results if r["ic_name"] == ic["name"]]
        if not ic_results:
            continue

        print(f"\n🔹 {ic['label']} ({ic['name']})")
        print(f"   Total inferences: {len(ic_results)}")

        avg_safety = sum(r["quality_scores"].get("safety_score", 0) for r in ic_results) / len(ic_results)
        avg_relevance = sum(r["quality_scores"].get("relevance_score", 0) for r in ic_results) / len(ic_results)
        avg_professional = sum(r["quality_scores"].get("professional_tone_score", 0) for r in ic_results) / len(ic_results)
        avg_latency = sum(r["latency_ms"] for r in ic_results) / len(ic_results)

        print(f"\n   Average Quality Scores:")
        print(f"     Safety:            {avg_safety:.3f}")
        print(f"     Relevance:         {avg_relevance:.3f}")
        print(f"     Professional Tone: {avg_professional:.3f}")
        print(f"\n   Average Latency: {avg_latency:.0f}ms")

        print(f"\n   Threshold Status:")
        print(f"     {'✅' if avg_safety >= THRESHOLDS['safety_min'] else '❌'} Safety "
              f"({avg_safety:.3f} vs >= {THRESHOLDS['safety_min']})")
        print(f"     {'✅' if avg_relevance >= THRESHOLDS['relevance_min'] else '❌'} Relevance "
              f"({avg_relevance:.3f} vs >= {THRESHOLDS['relevance_min']})")
        print(f"     {'✅' if avg_latency <= THRESHOLDS['latency_max_ms'] else '❌'} Latency "
              f"({avg_latency:.0f}ms vs <= {THRESHOLDS['latency_max_ms']}ms)")

print("\n" + "=" * 80)
print("🎉 SETUP COMPLETE!")
print("=" * 80)
print(f"\n📊 Grafana Dashboard:\n   {dashboard_url}")
print(f"\n📝 CloudWatch Logs:")
for ic in INFERENCE_COMPONENTS:
    print(f"   {ic['label']}: {log_groups[ic['name']]}")
print(f"\n📈 CloudWatch Metrics:")
print(f"   Namespace: {METRIC_NAMESPACE}")
print(f"   Available metrics:")
for m in (
    "safety_score",
    "relevance_score",
    "professional_tone_score",
    "composite_quality_score",
    "quality_evaluation_latency",
):
    print(f"     - {m}")
print(f"\n🔔 Alerting:")
print(f"   Alert rules:    {grafana_url}/alerting/list")
print(f"   Notifications:  {grafana_url}/alerting/notifications")
print("\n" + "=" * 80)

## Step 14: Clean Up Resources

Run this section when you are finished exploring the sample. It deletes the AWS resources this notebook created so they stop incurring charges. Each deletion is best-effort — failures are logged but do not raise — so the cell is safe to re-run if some resources have already been removed.

**What gets deleted:**

- The 3 Grafana alert rules + the `LLM Quality Alerts` folder
- The Grafana SNS contact point + its notification-policy route (if SNS was configured)
- The Grafana dashboard + the `CloudWatch-Quality` data source
- The SNS topic + email subscriptions (if SNS was configured)
- The `GrafanaSNSPublish` inline IAM policy (if SNS was configured)
- The CloudWatch log groups created in Step 2

**Left in place by default** (re-runs reuse them; flip the flag below to delete the workspace):

- The Amazon Managed Grafana workspace itself
- The `AmazonGrafanaCloudWatchRole` IAM role and its base `CloudWatchReadAccess` policy
- CloudWatch metric data (ages out per the standard CloudWatch retention schedule)

> ⚠️  **Data loss warning.** Deleting CloudWatch log groups permanently removes the prompt and response text logged during Step 2. If you need the data for audit, export it to Amazon S3 (via a CloudWatch Logs subscription or `aws logs create-export-task`) **before** running this cell.

In [ ]:
# Best-effort cleanup. Re-runnable: each step catches its own failures.

# ⚠️  Set to True only if no other workloads share this Grafana workspace.
# Deleting the workspace removes ALL dashboards, alerts, and contact points
# in it — not only the ones this notebook created.
DELETE_WORKSPACE = False

# Refresh an admin API key — earlier ones may have expired.
try:
    cleanup_key = grafana.create_workspace_api_key(
        workspaceId=workspace_id,
        keyName=f"cleanup-{int(time.time())}",
        keyRole="ADMIN",
        secondsToLive=1800,
    )["key"]
    cleanup_headers = {
        "Authorization": f"Bearer {cleanup_key}",
        "Content-Type": "application/json",
        "X-Disable-Provenance": "true",
    }
except Exception as e:
    print(f"⚠️  Could not refresh Grafana API key: {e}")
    cleanup_headers = None

if cleanup_headers:
    # 1. Delete the 3 alert rules
    for rule_uid in ("safety-score-alert", "relevance-score-alert", "composite-quality-alert"):
        try:
            requests.delete(
                f"{grafana_url}/api/v1/provisioning/alert-rules/{rule_uid}",
                headers=cleanup_headers,
            )
        except Exception as e:
            print(f"⚠️  Could not delete alert {rule_uid}: {e}")
    print("✅ Alert rules deletion attempted")

    # 2. Delete the alert folder (and any rules still in it)
    try:
        requests.delete(
            f"{grafana_url}/api/folders/{ALERT_FOLDER_UID}?forceDeleteRules=true",
            headers=cleanup_headers,
        )
        print(f"✅ Deleted alert folder: {ALERT_FOLDER_UID}")
    except Exception as e:
        print(f"⚠️  Could not delete alert folder: {e}")

    # 3. Delete the SNS contact point (only if Step 12 ran)
    try:
        requests.delete(
            f"{grafana_url}/api/v1/provisioning/contact-points/llm-quality-sns",
            headers=cleanup_headers,
        )
        print("✅ Deleted Grafana contact point: llm-quality-sns")
    except Exception as e:
        print(f"⚠️  Could not delete contact point: {e}")

    # 4. Reset the notification policy (drops our route)
    try:
        requests.delete(
            f"{grafana_url}/api/v1/provisioning/policies",
            headers=cleanup_headers,
        )
        print("✅ Reset notification policy")
    except Exception as e:
        print(f"⚠️  Could not reset notification policy: {e}")

    # 5. Delete the dashboard
    try:
        requests.delete(
            f"{grafana_url}/api/dashboards/uid/{GRAFANA_DASHBOARD_UID}",
            headers=cleanup_headers,
        )
        print(f"✅ Deleted dashboard: {GRAFANA_DASHBOARD_UID}")
    except Exception as e:
        print(f"⚠️  Could not delete dashboard: {e}")

    # 6. Delete the CloudWatch-Quality data source (look up its UID by name)
    try:
        ds_resp = requests.get(
            f"{grafana_url}/api/datasources/name/{GRAFANA_DATASOURCE_NAME}",
            headers=cleanup_headers,
        ).json()
        if isinstance(ds_resp, dict) and ds_resp.get("uid"):
            requests.delete(
                f"{grafana_url}/api/datasources/uid/{ds_resp['uid']}",
                headers=cleanup_headers,
            )
            print(f"✅ Deleted data source: {GRAFANA_DATASOURCE_NAME}")
    except Exception as e:
        print(f"⚠️  Could not delete data source: {e}")

# 7. Delete the SNS topic (also removes subscriptions). Only runs if Step 12 created one.
if 'sns_topic_arn' in dir():
    try:
        sns.delete_topic(TopicArn=sns_topic_arn)
        print(f"✅ Deleted SNS topic: {sns_topic_arn}")
    except Exception as e:
        print(f"⚠️  Could not delete SNS topic: {e}")

# 8. Detach the GrafanaSNSPublish inline policy (the role itself is shared
#    with other samples — don't delete it).
try:
    iam.delete_role_policy(
        RoleName=GRAFANA_ROLE_NAME,
        PolicyName="GrafanaSNSPublish",
    )
    print("✅ Removed GrafanaSNSPublish inline policy")
except iam.exceptions.NoSuchEntityException:
    pass
except Exception as e:
    print(f"⚠️  Could not remove GrafanaSNSPublish policy: {e}")

# 9. Delete the CloudWatch log groups created in Step 2.
for ic in INFERENCE_COMPONENTS:
    name = log_groups.get(ic['name'])
    if not name:
        continue
    try:
        cloudwatch_logs.delete_log_group(logGroupName=name)
        print(f"✅ Deleted log group: {name}")
    except cloudwatch_logs.exceptions.ResourceNotFoundException:
        pass
    except Exception as e:
        print(f"⚠️  Could not delete log group {name}: {e}")

# 10. Optionally delete the Grafana workspace itself.
if DELETE_WORKSPACE:
    try:
        grafana.delete_workspace(workspaceId=workspace_id)
        print(f"✅ Deleted Grafana workspace: {workspace_id}")
    except Exception as e:
        print(f"⚠️  Could not delete workspace: {e}")
else:
    print(f"\nℹ️  Workspace {workspace_id} left in place. Set DELETE_WORKSPACE=True to remove it.")

print("\n✅ Cleanup pass complete.")

## Next Steps

1. **View Dashboard**: Open the Grafana dashboard URL above to see quality metrics
2. **Configure Notifications**: The notebook deploys 3 quality threshold alert rules. To receive notifications when they fire, configure a contact point in the Grafana UI at `/alerting/notifications`. Amazon Managed Grafana supports SNS, Slack, PagerDuty, OpsGenie, and VictorOps integrations. For email, route alerts to an Amazon SNS topic and subscribe your email address(es) to the topic — Managed Grafana does not natively send email.
3. **Continuous Monitoring**: Run this pipeline continuously or on a schedule
4. **Customize Metrics**: Add more evaluation criteria based on your use case
5. **Integration**: Integrate with CI/CD for automated quality checks

### CloudWatch Logs Insights Queries:

**View all inferences:**
```
fields @timestamp, type, latency_ms, prompt, response
| filter type = "inference"
| sort @timestamp desc
```

**View quality scores:**
```
fields @timestamp, scores.`safety/mean`, scores.`relevance_to_query/mean`, scores.`professional_tone/mean`
| filter type = "quality_evaluation"
| sort @timestamp desc
```

**View quality scores with all fields:**
```
fields @timestamp, scores
| filter type = "quality_evaluation"
| sort @timestamp desc
```